In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Verify model exists
import os
model_path = '/content/drive/MyDrive/BoneDisease_Project/models/best_model_new.pth'
if os.path.exists(model_path):
    print(f"✅ Model found at: {model_path}")
else:
    print(f"❌ Model NOT found! Check path: {model_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model found at: /content/drive/MyDrive/BoneDisease_Project/models/best_model_new.pth


In [3]:
!pip install streamlit torch torchvision pillow numpy opencv-python transformers pyngrok -q
print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 137.7 MB/s eta 0:00:00
✅ Packages installed


In [4]:
%%writefile app_real.py
import streamlit as st
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import cv2
from PIL import Image
import time
import os
from transformers import AutoModel

# Page config
st.set_page_config(
    page_title="Bone Disease Classifier",
    page_icon="🦴",
    layout="wide"
)

st.title("🦴 Knee X-ray Bone Disease Classifier")
st.markdown("---")

# ============================================
# LOAD MODELS
# ============================================
@st.cache_resource
def load_models():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load DINOv2
    dinov2 = AutoModel.from_pretrained('facebook/dinov2-base').half().to(device)
    dinov2.eval()

    # MIL Model Architecture
    class MILModel(nn.Module):
        def __init__(self, input_dim=2304, hidden_dim=512, num_classes=3):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, hidden_dim)
            self.bn1 = nn.BatchNorm1d(hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
            self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
            self.fc3 = nn.Linear(hidden_dim // 2, num_classes)
            self.dropout = nn.Dropout(0.3)
            self.relu = nn.ReLU()

        def forward(self, x):
            x = self.dropout(self.relu(self.bn1(self.fc1(x))))
            x = self.dropout(self.relu(self.bn2(self.fc2(x))))
            return self.fc3(x)

    # Load trained model - VERIFY THIS PATH
    model = MILModel().to(device)
    model_path = '/content/drive/MyDrive/BoneDisease_Project/models/best_model_new.pth'

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        st.success(f"✅ Model loaded from Drive")
    else:
        st.error(f"❌ Model not found at {model_path}")
        return None, None, None

    return dinov2, model, device

# ============================================
# FEATURE EXTRACTION
# ============================================
def extract_features(image_path, dinov2, device):
    image = cv2.imread(image_path)
    if image is None:
        image = np.array(Image.open(image_path).convert('RGB'))
    else:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]
    patches = []
    patch_size, stride = 224, 112

    # Extract patches
    if h < patch_size or w < patch_size:
        image = cv2.resize(image, (patch_size, patch_size))
        patches = [image]
    else:
        for y in range(0, h - patch_size + 1, stride):
            for x in range(0, w - patch_size + 1, stride):
                patches.append(image[y:y+patch_size, x:x+patch_size])
        center_y = max(0, h//2 - patch_size//2)
        center_x = max(0, w//2 - patch_size//2)
        patches.append(image[center_y:center_y+patch_size, center_x:center_x+patch_size])

    patches = patches[:30]

    # Process patches
    patch_features = []
    for i in range(0, len(patches), 4):
        batch_patches = patches[i:i+4]
        batch_tensor = []

        for patch in batch_patches:
            if patch.shape[:2] != (224, 224):
                patch = cv2.resize(patch, (224, 224))
            patch_tensor = torch.from_numpy(patch).float().permute(2, 0, 1) / 255.0
            patch_tensor = transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )(patch_tensor)
            batch_tensor.append(patch_tensor)

        if batch_tensor:
            batch_tensor = torch.stack(batch_tensor).half().to(device)
            with torch.no_grad():
                outputs = dinov2(batch_tensor)
                features = outputs.last_hidden_state[:, 0, :].cpu().float().numpy()
                patch_features.append(features)

    if patch_features:
        patch_features = np.concatenate(patch_features, axis=0)
        max_features = np.max(patch_features, axis=0)
        mean_features = np.mean(patch_features, axis=0)
        attention_weights = np.std(patch_features, axis=1)
        attention_weights = attention_weights / (np.sum(attention_weights) + 1e-8)
        attention_features = np.sum(patch_features * attention_weights[:, np.newaxis], axis=0)
        combined_features = np.concatenate([max_features, mean_features, attention_features])
        return combined_features

    return None

# ============================================
# PREDICTION FUNCTION
# ============================================
def predict_image(image_path, dinov2, model, device):
    features = extract_features(image_path, dinov2, device)
    if features is None:
        return None, None, None

    with torch.no_grad():
        input_tensor = torch.FloatTensor(features).unsqueeze(0).to(device)
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)[0].cpu().numpy()
        predicted_class = np.argmax(probabilities)
        confidence = probabilities[predicted_class]

    return predicted_class, confidence, probabilities

# ============================================
# MAIN APP
# ============================================
with st.spinner("🔄 Loading AI models... (This takes 30-60 seconds)"):
    dinov2, model, device = load_models()

if dinov2 is None:
    st.stop()

CLASS_NAMES = ['Normal', 'Osteopenia', 'Osteoporosis']

# Sidebar
with st.sidebar:
    st.header("📋 Model Info")
    st.info(f"""
    - **Accuracy**: 89.38%
    - **Classes**: Normal, Osteopenia, Osteoporosis
    - **Device**: {device}
    """)
    st.markdown("---")
    st.header("📤 Upload")
    uploaded_file = st.file_uploader("Choose a knee X-ray", type=['jpg', 'jpeg', 'png'])

# Main content
col1, col2 = st.columns(2)

if uploaded_file is not None:
    with open("temp_image.jpg", "wb") as f:
        f.write(uploaded_file.getbuffer())

    image = Image.open(uploaded_file)
    with col1:
        st.subheader("📸 Uploaded X-ray")
        st.image(image, use_container_width=True)

        if st.button("🔍 Analyze Image", type="primary"):
            with st.spinner("🔬 Analyzing with AI model..."):
                pred_class, confidence, probabilities = predict_image("temp_image.jpg", dinov2, model, device)

                if pred_class is not None:
                    with col2:
                        st.subheader("📊 Results")

                        if pred_class == 0:
                            color, emoji, bg = "green", "✅", "#d4edda"
                        elif pred_class == 1:
                            color, emoji, bg = "orange", "⚠️", "#fff3cd"
                        else:
                            color, emoji, bg = "red", "🔴", "#f8d7da"

                        st.markdown(f"""
                        <div style='padding: 20px; border-radius: 10px; background-color: {bg};'>
                            <h2 style='color: {color};'>{emoji} {CLASS_NAMES[pred_class]}</h2>
                            <h3>Confidence: {confidence*100:.1f}%</h3>
                        </div>
                        """, unsafe_allow_html=True)

                        st.subheader("📈 Class Probabilities")
                        for i, cls in enumerate(CLASS_NAMES):
                            st.progress(float(probabilities[i]), text=f"{cls}: {probabilities[i]*100:.1f}%")
                else:
                    st.error("❌ Failed to analyze image")
else:
    with col1:
        st.info("👆 Upload an X-ray image to begin")
    with col2:
        st.info("📊 Results will appear here")

st.markdown("---")
st.markdown("© 2024 Bone Disease Classifier | Using your actual 89.38% accurate model")

Writing app_real.py


In [24]:
import os
import time

# Kill old Streamlit process
os.system('pkill -f streamlit')

# Wait a moment
time.sleep(2)

# Start Streamlit fresh with the new app
os.system('streamlit run app_real.py --server.port 8502 --server.headless true &')

print("✅ Streamlit restarted with new app")
print("⏳ Wait 10 seconds then refresh your browser")

✅ Streamlit restarted with new app
⏳ Wait 10 seconds then refresh your browser


In [25]:
import requests
import time

# Get your Colab's external IP
ip = requests.get('https://api.ipify.org').text
print("="*60)
print("🚀 TRY THESE URLs IN YOUR BROWSER:")
print("="*60)
print(f"1️⃣  Local URL: http://localhost:8502")
print(f"2️⃣  External URL: http://{ip}:8502")
print(f"3️⃣  Colab URL: https://{ip}:8502")
print("="*60)
print("\n👉 Click the External URL first")
print("👉 If it doesn't work, use serveo.net method below")

🚀 TRY THESE URLs IN YOUR BROWSER:
1️⃣  Local URL: http://localhost:8502
2️⃣  External URL: http://34.125.142.173:8502
3️⃣  Colab URL: https://34.125.142.173:8502

👉 Click the External URL first
👉 If it doesn't work, use serveo.net method below


In [26]:
import threading
import time

def run_tunnel():
    !ssh -o StrictHostKeyChecking=no -R 80:localhost:8502 serveo.net

# Start tunnel
thread = threading.Thread(target=run_tunnel)
thread.start()

print("="*60)
print("🚀 WAIT FOR THE URL TO APPEAR ABOVE")
print("="*60)
print("\n⏳ Look for a line like:")
print("   'Forwarding HTTP traffic from https://abc123.serveo.net'")
print("\n👉 Copy and open that URL")
print("="*60)

# Keep alive
while True:
    time.sleep(30)
    print(f"🔄 Tunnel running...")

🚀 WAIT FOR THE URL TO APPEAR ABOVE

⏳ Look for a line like:
   'Forwarding HTTP traffic from https://abc123.serveo.net'

👉 Copy and open that URL
Forwarding HTTP traffic from https://cc13d95864ff20a8-34-125-142-173.serveousercontent.com
🔄 Tunnel running...


KeyboardInterrupt: 